In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT & ÉP PYTHON NHẬN DIỆN THƯ MỤC
# =========================================================
import os
import sys
import json
import csv
from types import ModuleType

# Nếu cần dùng NMS
import torch
from torchvision.ops import nms

# Làm việc trong /kaggle/working
%cd /kaggle/working

# Clone DocLayout-YOLO nếu chưa có
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd DocLayout-YOLO
!pip install -q -e .

# Ép repo vào sys.path
repo_dir = "/kaggle/working/DocLayout-YOLO"
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

%cd /kaggle/working

# =========================================================
# BƯỚC 2: "HACK" BỘ NHỚ ĐỂ FIX LỖI MODULE HUB
# =========================================================
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 39.17 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.6 MB/s eta 0:00:00
  Building editable for doclayout_yolo (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 

In [2]:
from pathlib import Path
import shutil
import torch
from torchvision.ops import nms
from doclayout_yolo import YOLOv10

print("✅ Import YOLOv10 thành công!")

# ---------------------------------------------------------
# HÀM HẬU XỬ LÝ BOX CHO TỪNG MÔ HÌNH (Giữ nguyên logic cũ, thêm trả về conf)
# ---------------------------------------------------------
def postprocess_boxes(r,
                      img_w: int,
                      img_h: int,
                      names,
                      iou_thr: float = 0.6,
                      pad_scale_x: float = 0.0,
                      pad_scale_y: float = 0.0):
    boxes = []
    scores = []
    labels = []

    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])

        boxes.append([x1, y1, x2, y2])
        scores.append(conf)
        labels.append(cls_id)

    if len(boxes) == 0:
        return []

    boxes = torch.tensor(boxes)
    scores = torch.tensor(scores)
    labels = torch.tensor(labels)

    keep_indices = []
    unique_classes = set(labels.tolist())
    for cls in unique_classes:
        cls_mask = labels == cls
        cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
        cls_boxes = boxes[cls_indices]
        cls_scores = scores[cls_indices]

        kept = nms(cls_boxes, cls_scores, iou_thr)
        keep_indices.extend(cls_indices[kept].tolist())

    keep_indices = sorted(set(keep_indices))

    regions = []
    for idx in keep_indices:
        x1, y1, x2, y2 = boxes[idx].tolist()
        h = y2 - y1

        pad_x = pad_scale_x * h
        pad_y = pad_scale_y * h

        x1 = max(0.0, x1 - pad_x)
        y1 = max(0.0, y1 - pad_y)
        x2 = min(float(img_w), x2 + pad_x)
        y2 = min(float(img_h), y2 + pad_y)

        cls_id = int(labels[idx])
        cls_name = names[cls_id] if names is not None else "region"
        conf = float(scores[idx])

        # Trả về kèm conf để phục vụ Ensemble
        regions.append({
            "bbox": [x1, y1, x2, y2],
            "type": cls_name,
            "conf": conf,
            "text": ""
        })

    return regions

# ---------------------------------------------------------
# HÀM ENSEMBLE BOXES TỪ NHIỀU MÔ HÌNH
# ---------------------------------------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

def ensemble_boxes(all_model_preds, iou_thr=0.55):
    # Sắp xếp tất cả dự đoán giảm dần theo conf
    all_model_preds = sorted(all_model_preds, key=lambda x: x['conf'], reverse=True)
    
    clusters = []
    for pred in all_model_preds:
        box = pred['bbox']
        matched = False
        
        for cluster in clusters:
            # So sánh IoU bất chấp nhãn (để xử lý vụ trùng box khác nhãn)
            if compute_iou(box, cluster['avg_bbox']) > iou_thr:
                cluster['elements'].append(pred)
                
                # Cập nhật lại trung bình tọa độ (Weighted Average bằng Conf)
                sum_conf = sum(e['conf'] for e in cluster['elements'])
                new_x1 = sum(e['bbox'][0] * e['conf'] for e in cluster['elements']) / sum_conf
                new_y1 = sum(e['bbox'][1] * e['conf'] for e in cluster['elements']) / sum_conf
                new_x2 = sum(e['bbox'][2] * e['conf'] for e in cluster['elements']) / sum_conf
                new_y2 = sum(e['bbox'][3] * e['conf'] for e in cluster['elements']) / sum_conf
                
                cluster['avg_bbox'] = [new_x1, new_y1, new_x2, new_y2]
                matched = True
                break
                
        if not matched:
            clusters.append({
                'avg_bbox': box,
                'elements': [pred]
            })

    final_regions = []
    for cluster in clusters:
        # 1. Chốt tọa độ trung bình
        final_bbox = [round(x, 2) for x in cluster['avg_bbox']]
        
        # 2. Vote chốt Label dựa trên tổng Conf
        class_scores = {}
        for e in cluster['elements']:
            lbl = e['type']
            class_scores[lbl] = class_scores.get(lbl, 0.0) + e['conf']
            
        final_label = max(class_scores, key=class_scores.get)
        
        final_regions.append({
            "bbox": final_bbox,
            "type": final_label,
            "text": ""
        })
        
    return final_regions

✅ Import YOLOv10 thành công!


In [3]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ---------------------------------------------------------
import json
import csv

# Danh sách 3 Models (3 Folds)
MODEL_PATHS = [
    "/kaggle/input/models/notpitomon/doclayoutyolo-v5/pytorch/default/2/DocLayoutYOLO-v5.1/DLYv5.1 - 1.pt",
    "/kaggle/input/models/notpitomon/doclayoutyolo-v5/pytorch/default/2/DocLayoutYOLO-v5.1/DLYv5.1 - 2.pt",
    "/kaggle/input/models/notpitomon/doclayoutyolo-v5/pytorch/default/2/DocLayoutYOLO-v5.1/DLYv5.1 - 3.pt",
    "/kaggle/input/models/notpitomon/doclayoutyolo-v5/pytorch/default/2/DocLayoutYOLO-v5.1/DLYv5.1 - 4.pt",
    "/kaggle/input/models/notpitomon/doclayoutyolo-v5/pytorch/default/2/DocLayoutYOLO-v5.1/DLYv5.1 - 5.pt"
]

TEST_IMG_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images")

image_paths = []
if TEST_IMG_DIR.exists():
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(TEST_IMG_DIR.glob(ext))
else:
    raise FileNotFoundError(f"Không tìm thấy thư mục: {TEST_IMG_DIR}")

image_paths = sorted(image_paths)

OUT_DIR = Path("/kaggle/working/inference_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = OUT_DIR / "metadata.jsonl"
SUB_PATH = OUT_DIR / "submission.csv"

# Tham số YOLO
IMG_SIZE = 1280
CONF = 0.15
MAX_DET = 200

# Tham số hậu xử lý riêng lẻ
IOU_NMS = 0.6
PAD_SCALE_X = 0.0
PAD_SCALE_Y = 0.0

# Tham số Ensemble Bboxes giữa các models
ENSEMBLE_IOU_THR = 0.55

# Khởi tạo 3 model vào RAM/VRAM
print("\nĐang load 3 models...")
models = []
for mp in MODEL_PATHS:
    assert Path(mp).exists(), f"Không tìm thấy model: {mp}"
    models.append(YOLOv10(mp))
print("✅ Load xong 5 models!")

print("🚀 Bắt đầu trích xuất tọa độ Ensemble & sinh submission.csv...")

count_images = 0

with open(META_PATH, 'w', encoding='utf-8') as f_meta, \
     open(SUB_PATH, 'w', newline='', encoding='utf-8') as f_sub:

    writer = csv.writer(f_sub)
    writer.writerow(["image", "regions"])

    for img_path in image_paths:
        all_models_regions = []
        img_w, img_h = 0, 0
        
        # Chạy inference qua từng model
        for model in models:
            results = model.predict(
                source=str(img_path),
                imgsz=IMG_SIZE,
                conf=CONF,
                max_det=MAX_DET,
                verbose=False
            )
            r = results[0]
            img_h, img_w = r.orig_shape

            # Tiền xử lý riêng lẻ (Giữ nguyên logic NMS per-class)
            regions = postprocess_boxes(
                r,
                img_w=img_w,
                img_h=img_h,
                names=model.names,
                iou_thr=IOU_NMS,
                pad_scale_x=PAD_SCALE_X,
                pad_scale_y=PAD_SCALE_Y
            )
            all_models_regions.extend(regions)

        # Gộp chung kết quả từ 3 Folds
        final_regions = ensemble_boxes(all_models_regions, iou_thr=ENSEMBLE_IOU_THR)

        # Ghi file
        data_record = {
            "file_name": f"images/{img_path.name}",
            "image_width": img_w,
            "image_height": img_h,
            "annotation_source": "yolov10_ensemble",
            "regions": final_regions
        }
        f_meta.write(json.dumps(data_record, ensure_ascii=False) + "\n")

        writer.writerow([img_path.name, json.dumps(final_regions, ensure_ascii=False)])

        count_images += 1
        if count_images % 50 == 0:
            print(f"⏳ Đã xử lý {count_images}/{len(image_paths)} ảnh...")

print("\n--- HOÀN TẤT ---")
print(f"Đã xử lý thành công: {count_images} ảnh")
print(f"File metadata: {META_PATH}")
print(f"File nộp Kaggle: {SUB_PATH}")


Đang load 3 models...
✅ Load xong 5 models!
🚀 Bắt đầu trích xuất tọa độ Ensemble & sinh submission.csv...
⏳ Đã xử lý 50/385 ảnh...
⏳ Đã xử lý 100/385 ảnh...
⏳ Đã xử lý 150/385 ảnh...
⏳ Đã xử lý 200/385 ảnh...
⏳ Đã xử lý 250/385 ảnh...
⏳ Đã xử lý 300/385 ảnh...
⏳ Đã xử lý 350/385 ảnh...

--- HOÀN TẤT ---
Đã xử lý thành công: 385 ảnh
File metadata: /kaggle/working/inference_output/metadata.jsonl
File nộp Kaggle: /kaggle/working/inference_output/submission.csv


In [4]:
# # ---------------------------------------------------------
# # CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# # ---------------------------------------------------------
# from pathlib import Path
# import shutil
# import json
# import csv
# import cv2
# import matplotlib.pyplot as plt
# from doclayout_yolo import YOLOv10

# # Đường dẫn weights của bạn (đổi lại cho đúng)
# MODEL_PATH = "/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/3/DoclayoutYoloV4.2.pt"

# # CHỈ ĐỊNH DUY NHẤT 2 ẢNH THEO YÊU CẦU
# image_paths = [
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images/477124d6-4236-5aa1-8ffe-2c1831ae4a8a.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images/8773aeed-1297-5372-b537-1097a2030e26.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/silver/images/6b79b26a-f1b7-49f4-91a9-043d9c8b884f.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images/47ca0b29-b0d7-517e-93ce-c8c9b7e3d698.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images/964a31fc-0bee-5096-9359-2371641869eb.jpg")
# ]

# # Lọc các file tồn tại
# image_paths = [p for p in image_paths if p.exists()]

# # Thư mục output
# OUT_DIR = Path("/kaggle/working/inference_output")
# if OUT_DIR.exists():
#     shutil.rmtree(OUT_DIR)
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # File metadata jsonl (debug) + file nộp Kaggle
# META_PATH = OUT_DIR / "metadata.jsonl"
# SUB_PATH = OUT_DIR / "submission.csv"

# # Tham số YOLO
# IMG_SIZE = 1280
# CONF = 0.15
# MAX_DET = 200

# # Tham số hậu xử lý
# IOU_NMS = 0.6
# PAD_SCALE_X = 0.0
# PAD_SCALE_Y = 0.0

# assert Path(MODEL_PATH).exists(), f"Không tìm thấy model: {MODEL_PATH}"
# print(f"📸 Tìm thấy tổng cộng {len(image_paths)} ảnh cần dự đoán.")

# # Khởi tạo model
# print("\nĐang load model...")
# model = YOLOv10(MODEL_PATH)

# print("🚀 Bắt đầu dự đoán & vẽ ảnh...")

# count_images = 0

# with open(META_PATH, 'w', encoding='utf-8') as f_meta, \
#      open(SUB_PATH, 'w', newline='', encoding='utf-8') as f_sub:

#     writer = csv.writer(f_sub)
#     writer.writerow(["image", "regions"])

#     for img_path in image_paths:
#         # Chạy dự đoán
#         results = model.predict(
#             source=str(img_path),
#             imgsz=IMG_SIZE,
#             conf=CONF,
#             max_det=MAX_DET,
#             verbose=False
#         )

#         r = results[0]
#         img_h, img_w = r.orig_shape

#         # Hậu xử lý: NMS + padding
#         regions = postprocess_boxes(
#             r,
#             img_w=img_w,
#             img_h=img_h,
#             names=model.names,
#             iou_thr=IOU_NMS,
#             pad_scale_x=PAD_SCALE_X,
#             pad_scale_y=PAD_SCALE_Y
#         )

#         # ---------------------------------------------------------
#         # VẼ BBOX VÀ CLASS LÊN ẢNH SAU ĐÓ HIỂN THỊ
#         # ---------------------------------------------------------
#         img_vis = cv2.imread(str(img_path))
#         img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

#         for region in regions:
#             x1, y1, x2, y2 = map(int, map(round, region["bbox"]))
#             cls_name = region["type"]

#             # Vẽ bounding box màu đỏ (Red)
#             cv2.rectangle(img_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
#             # Ghi text (tên class) ngay trên góc trái của bbox
#             cv2.putText(
#                 img_vis, cls_name, (x1, max(y1 - 10, 0)), 
#                 cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 0), 3
#             )

#         # Hiển thị ảnh ngay trên notebook
#         plt.figure(figsize=(16, 16))
#         plt.imshow(img_vis)
#         plt.axis("off")
#         plt.title(f"Kết quả cho: {img_path.name}")
#         plt.show()
#         # ---------------------------------------------------------

#         # Ghi metadata.jsonl để debug
#         data_record = {
#             "file_name": f"images/{img_path.name}",
#             "image_width": img_w,
#             "image_height": img_h,
#             "annotation_source": "yolov10_prediction",
#             "regions": regions
#         }
#         f_meta.write(json.dumps(data_record, ensure_ascii=False) + "\n")

#         # Ghi 1 dòng vào submission.csv
#         writer.writerow([img_path.name, json.dumps(regions, ensure_ascii=False)])

#         count_images += 1

# print("\n--- HOÀN TẤT ---")
# print(f"Đã xử lý và vẽ thành công: {count_images} ảnh")
# print(f"File metadata: {META_PATH}")
# print(f"File nộp Kaggle: {SUB_PATH}")

In [5]:
import pandas as pd

df_sub = pd.read_csv(SUB_PATH)
display(df_sub.head())

print("\nSố dòng trong submission:", len(df_sub))
print("Cột:", list(df_sub.columns))

,image,regions
0,002b94ef-e000-4e76-bc7e-7846166bc806.jpg,"[{""bbox"": [547.75, 833.87, 3804.72, 1062.47], ..."
1,00829c22-1e80-4e13-a395-3f9d8b088f7d.jpg,"[{""bbox"": [285.47, 1301.8, 2816.96, 1441.75], ..."
2,00af1ec3-5315-4782-bc61-bf498617e5a2.jpg,"[{""bbox"": [60.61, 741.3, 1591.52, 868.93], ""ty..."
3,010c6dad-2f5d-4e4c-802b-f978a7e1998b.jpg,"[{""bbox"": [1224.05, 2309.66, 3839.8, 2630.48],..."
4,011934b4-d919-5000-8c1e-39f96daeba68.jpg,"[{""bbox"": [1.13, 1589.1, 1485.07, 1745.21], ""t..."



Số dòng trong submission: 385
Cột: ['image', 'regions']
